In [56]:
#train.csv로 모델을 학습, 학습한 모델을 test.csv에 적용
#bank marketing 참여 여부(정기 예금 가입 여부)를 예측하는 classification tast.

In [57]:
#특성 엔지니어링 (변수 손질)

In [58]:
import warnings
warnings.filterwarnings('ignore')

In [59]:
#!pip install catboost

In [60]:
import numpy as np
import os
import pandas as pd
from catboost import CatBoostClassifier

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV

from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
import lightgbm as lgbm
from catboost import CatBoostClassifier
from sklearn.ensemble import BaggingClassifier, VotingClassifier

In [61]:
df_trn = pd.read_csv('train.csv')
df_tst=pd.read_csv('test.csv')

In [62]:
df_trn.head()

,ID,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,label
0,train00001,34,blue-collar,married,primary,no,358,yes,no,unknown,23,may,100,4,-1,0,unknown,0
1,train00002,33,blue-collar,married,secondary,no,-53,yes,no,unknown,20,may,172,1,-1,0,unknown,0
2,train00003,32,management,single,tertiary,no,207,yes,no,cellular,2,feb,56,1,-1,0,unknown,0
3,train00004,37,blue-collar,divorced,secondary,no,638,yes,no,cellular,8,may,326,1,326,2,success,0
4,train00005,33,housemaid,married,secondary,no,826,yes,no,cellular,11,may,256,1,-1,0,unknown,0


In [63]:
df_trn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31647 entries, 0 to 31646
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         31647 non-null  object
 1   age        31647 non-null  int64 
 2   job        31647 non-null  object
 3   marital    31647 non-null  object
 4   education  31647 non-null  object
 5   default    31647 non-null  object
 6   balance    31647 non-null  int64 
 7   housing    31647 non-null  object
 8   loan       31647 non-null  object
 9   contact    31647 non-null  object
 10  day        31647 non-null  int64 
 11  month      31647 non-null  object
 12  duration   31647 non-null  int64 
 13  campaign   31647 non-null  int64 
 14  pdays      31647 non-null  int64 
 15  previous   31647 non-null  int64 
 16  poutcome   31647 non-null  object
 17  label      31647 non-null  int64 
dtypes: int64(8), object(10)
memory usage: 4.3+ MB


In [64]:
month_map = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

df_trn['month'] = df_trn['month'].map(month_map)
df_tst['month'] = df_tst['month'].map(month_map)

In [65]:
df_trn.head()

,ID,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,label
0,train00001,34,blue-collar,married,primary,no,358,yes,no,unknown,23,5,100,4,-1,0,unknown,0
1,train00002,33,blue-collar,married,secondary,no,-53,yes,no,unknown,20,5,172,1,-1,0,unknown,0
2,train00003,32,management,single,tertiary,no,207,yes,no,cellular,2,2,56,1,-1,0,unknown,0
3,train00004,37,blue-collar,divorced,secondary,no,638,yes,no,cellular,8,5,326,1,326,2,success,0
4,train00005,33,housemaid,married,secondary,no,826,yes,no,cellular,11,5,256,1,-1,0,unknown,0


In [66]:

print(df_trn['job'].unique())

['blue-collar' 'management' 'housemaid' 'admin.' 'technician' 'services'
 'retired' 'student' 'unknown' 'unemployed' 'entrepreneur' 'self-employed']


In [67]:
y_cols=['label']
num_cols=['age', 'balance', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous']
cat_cols=['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']
id_cols=['ID']
x_cols=num_cols+cat_cols

In [68]:
train_corr=df_trn[num_cols].corr()
high_corr=train_corr[train_corr.abs()>=0.3]
print(high_corr)
#0.4에서 0.6이면 보통의 상관관계라 애매.

          age  balance  day  month  duration  campaign     pdays  previous
age       1.0      NaN  NaN    NaN       NaN       NaN       NaN       NaN
balance   NaN      1.0  NaN    NaN       NaN       NaN       NaN       NaN
day       NaN      NaN  1.0    NaN       NaN       NaN       NaN       NaN
month     NaN      NaN  NaN    1.0       NaN       NaN       NaN       NaN
duration  NaN      NaN  NaN    NaN       1.0       NaN       NaN       NaN
campaign  NaN      NaN  NaN    NaN       NaN       1.0       NaN       NaN
pdays     NaN      NaN  NaN    NaN       NaN       NaN  1.000000  0.432882
previous  NaN      NaN  NaN    NaN       NaN       NaN  0.432882  1.000000


In [69]:
df_trn_x = df_trn[x_cols]
df_tst_x = df_tst[x_cols]

df_trn_y = df_trn[y_cols]
df_tst_y = df_tst[y_cols]

In [70]:
# 1. 연령대(age) 범주화
df_trn_x['age_group'] = pd.cut(df_trn_x['age'], bins=[0, 30, 60, 100], labels=['young', 'middle_aged', 'senior'])
df_tst_x['age_group'] = pd.cut(df_tst_x['age'], bins=[0, 30, 60, 100], labels=['young', 'middle_aged', 'senior'])

# 2. 잔고(balance) 범주화
df_trn_x['balance_group'] = pd.cut(df_trn_x['balance'], bins=[-1000, 0, 5000, 10000, 50000], labels=['negative', 'low', 'medium', 'high'])
df_tst_x['balance_group'] = pd.cut(df_tst_x['balance'], bins=[-1000, 0, 5000, 10000, 50000], labels=['negative', 'low', 'medium', 'high'])

# 3. 캠페인 참여 기간(duration * campaign)
df_trn_x['campaign_duration'] = df_trn_x['duration'] * df_trn_x['campaign']
df_tst_x['campaign_duration'] = df_tst_x['duration'] * df_tst_x['campaign']

In [71]:
for col in cat_cols:
    df_trn_x[col] = df_trn_x[col].astype('category')
    df_tst_x[col] = df_tst_x[col].astype('category')

In [72]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# 범주형 변수+특성 엔지니어링한 변수를 Label Encoding으로 변환
cat_cols_with_new = cat_cols + ['age_group', 'balance_group'] 
label_encoders = {}  # 각 컬럼에 대한 LabelEncoder 객체 저장

for col in cat_cols_with_new:
    le = LabelEncoder()
    df_trn_x[col] = le.fit_transform(df_trn_x[col])  # 훈련 데이터 변환
    df_tst_x[col] = le.transform(df_tst_x[col])      # 테스트 데이터 변환
    label_encoders[col] = le

In [73]:
model = lgbm.LGBMClassifier(verbose=-1)

In [74]:
n_positive = sum(df_trn_y.iloc[:, 0] == 1)  # 양성 클래스의 샘플 수
n_negative = sum(df_trn_y.iloc[:, 0] == 0)
# scale_pos_weight 설정 (양성/음성 비율)
scale_pos_weight = n_negative / n_positive

In [75]:
param_grid = {
    'iterations': np.arange(100, 2001, 100),
    'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2],
    'l2_leaf_reg': [1, 3, 5, 10],
    'depth': np.arange(3, 10),
    'subsample': [0.8, 0.9, 1.0],
    'scale_pos_weight': [scale_pos_weight]
}

In [76]:
%%time
random_search = RandomizedSearchCV(estimator=model,
                                   param_distributions=param_grid, 
                                   scoring='f1', 
                                   cv=7, 
                                   refit=True)
random_search.fit(df_trn_x, df_trn_y.values.ravel())

CPU times: total: 4min 41s
Wall time: 1min 44s


RandomizedSearchCV(cv=7, estimator=LGBMClassifier(verbose=-1),
                   param_distributions={'depth': array([3, 4, 5, 6, 7, 8, 9]),
                                        'iterations': array([ 100,  200,  300,  400,  500,  600,  700,  800,  900, 1000, 1100,
       1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000]),
                                        'l2_leaf_reg': [1, 3, 5, 10],
                                        'learning_rate': [0.01, 0.03, 0.05,
                                                          0.07, 0.1, 0.15,
                                                          0.2],
                                        'scale_pos_weight': [7.548622366288493],
                                        'subsample': [0.8, 0.9, 1.0]},
                   scoring='f1')

In [77]:
best_model = random_search.best_estimator_

In [78]:
catboost_model = CatBoostClassifier(iterations=1000, 
                                    depth=10, 
                                    learning_rate=0.05, 
                                    scale_pos_weight=scale_pos_weight,
                                    loss_function='Logloss', 
                                    verbose=False)

In [79]:
bagging_catboost_model = BaggingClassifier(estimator=catboost_model,
                                          n_estimators=50, 
                                          random_state=42)

In [80]:
ensemble_model = VotingClassifier(estimators=[
    ('lightgbm', best_model),  # Bagging + LightGBM 모델
    ('catboost', catboost_model)  # Bagging + CatBoost 모델
], voting='soft') 

In [81]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
# 교차 검증을 통한 OOF 예측
n_splits = 5  # 원하는 fold 수
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [82]:
# OOF 예측값 초기화
oof_preds = np.zeros(df_trn_y.shape[0])

# 테스트 예측값 초기화 ← 반드시 for문 밖
test_preds = np.zeros(df_tst_x.shape[0])

for fold, (train_idx, val_idx) in enumerate(kf.split(df_trn_x, df_trn_y)):
    
    X_train, X_val = df_trn_x.iloc[train_idx], df_trn_x.iloc[val_idx]
    y_train, y_val = df_trn_y.iloc[train_idx], df_trn_y.iloc[val_idx]
    
    ensemble_model.fit(X_train, y_train)
    
    val_preds = ensemble_model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    
    # 테스트 예측 누적
    fold_test_preds = ensemble_model.predict_proba(df_tst_x)[:, 1]
    test_preds += fold_test_preds
    
    fold_f1 = f1_score(y_val, val_preds.round())
    print(f"Fold {fold + 1} F1: {fold_f1:.4f}")

# 5개 fold 평균
test_preds /= n_splits

# OOF 전체 성능
oof_f1 = f1_score(df_trn_y.values.ravel(), oof_preds.round())
print(f"Overall OOF F1: {oof_f1:.4f}")

Fold 1 F1: 0.6200
Fold 2 F1: 0.6092
Fold 3 F1: 0.6190
Fold 4 F1: 0.6371
Fold 5 F1: 0.6170
Overall OOF F1: 0.6204


In [83]:
df_trn_x['oof_preds'] = oof_preds
df_tst_x['test_preds'] = test_preds

In [84]:
# OOF와 Test 예측값은 따로 보관
oof_pred_prob = df_trn_x['oof_preds'].copy()
test_pred_prob = df_tst_x['test_preds'].copy()

In [85]:
X_train_final = df_trn_x.drop(columns=['oof_preds'])
X_test_final = df_tst_x.drop(columns=['test_preds'])

In [86]:
print("Train:", X_train_final.shape)
print("Test :", X_test_final.shape)

Train: (31647, 19)
Test : (13564, 19)


In [87]:
thresholds = np.arange(0.1, 0.91, 0.01)

best_threshold = 0
best_f1 = 0

for threshold in thresholds:
    oof_pred_label = (oof_pred_prob >= threshold).astype(int)
    score = f1_score(df_trn_y, oof_pred_label)
    
    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print(f"Best Threshold: {best_threshold:.2f}")
print(f"Best OOF F1 Score: {best_f1:.4f}")

Best Threshold: 0.52
Best OOF F1 Score: 0.6222


In [88]:
oof_final_pred = (oof_pred_prob >= best_threshold).astype(int)

In [89]:
final_f1 = f1_score(df_trn_y, oof_final_pred)
final_recall = recall_score(df_trn_y, oof_final_pred)
final_accuracy = accuracy_score(df_trn_y, oof_final_pred)

print(f"최종 OOF F1 Score : {final_f1:.4f}")
print(f"최종 OOF Recall   : {final_recall:.4f}")
print(f"최종 OOF Accuracy : {final_accuracy:.4f}")

최종 OOF F1 Score : 0.6222
최종 OOF Recall   : 0.7515
최종 OOF Accuracy : 0.8933


In [90]:
final_pred = (test_pred_prob >= best_threshold).astype(int)

In [91]:
df_submission = pd.read_csv('submission_example.csv')

In [92]:
df_submission.head()

,ID,label
0,test00001,0
1,test00002,0
2,test00003,0
3,test00004,0
4,test00005,0


In [93]:
df_submission['label'] = final_pred

In [94]:
df_submission

,ID,label
0,test00001,0
1,test00002,0
2,test00003,0
3,test00004,1
4,test00005,0
...,...,...
13559,test13560,1
13560,test13561,0
13561,test13562,0
13562,test13563,0


In [95]:
(df_submission['label']==1).sum() 

2218

In [96]:
filename = 'oof-csv'
df_submission.to_csv(filename, index=False)